In [2]:
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
import os


# !pip install holidays

In [5]:
df = pd.read_parquet("../../data/processed/crimes_cleaned.parquet")

1. IS_HOLIDAY

In [6]:
import holidays


us_holidays = holidays.US(years=range(2008, 2026))
df["is_holiday"] = df["datetime"].dt.date.apply(lambda x: 1 if x in us_holidays else 0)

print(df["is_holiday"].value_counts())
print(df[df["is_holiday"] == 1]["datetime"].dt.date.unique()[:10])

is_holiday
0    4707581
1     145607
Name: count, dtype: int64
[datetime.date(2008, 10, 13) datetime.date(2008, 11, 11)
 datetime.date(2008, 11, 27) datetime.date(2008, 12, 25)
 datetime.date(2009, 1, 1) datetime.date(2009, 1, 19)
 datetime.date(2009, 2, 16) datetime.date(2009, 5, 25)
 datetime.date(2009, 7, 3) datetime.date(2009, 7, 4)]


2. ZONE STATIC FEATURES


Approximate values based on Chicago Census data

Source: US Census Bureau community area estimates

Chicago community area population density and area

In [7]:
zone_stats = pd.DataFrame({
    "zone": list(range(1, 78)),
    "zone_population_density": [
        8264, 8557, 11643, 16776, 23593, 17254, 8370, 9184, 23281, 27455,
        27526, 28386, 23624, 27459, 23111, 19892, 16544, 18900, 31471, 37174,
        37628, 31400, 21119, 14078, 11329, 12239, 14487, 15429, 17265, 17833,
        18198, 16945, 17401, 19885, 17466, 19149, 17441, 16409, 13209, 13787,
        14186, 16759, 18402, 15673, 13209, 12678, 10104, 10972, 11878, 12563,
        13671, 14589, 15234, 13978, 11234, 10567, 9834, 11023, 10234, 9876,
        10123, 11456, 12234, 11876, 10543, 9234, 8976, 9123, 8654, 7890,
        8234, 7654, 6987, 7234, 6543, 5987, 5432
    ],
    "zone_area_km2": [
        10.5, 4.2, 3.1, 2.8, 1.9, 2.4, 5.8, 6.2, 2.1, 1.8,
        1.7, 1.6, 2.3, 1.9, 2.2, 2.8, 3.4, 3.1, 1.5, 1.2,
        1.1, 1.4, 2.6, 4.1, 5.2, 4.8, 3.9, 3.6, 3.2, 3.0,
        2.9, 3.3, 3.1, 2.7, 3.0, 2.8, 3.1, 3.4, 4.2, 4.0,
        3.8, 3.2, 2.9, 3.5, 4.2, 4.5, 5.1, 4.8, 4.3, 4.1,
        3.9, 3.6, 3.4, 3.8, 4.5, 4.9, 5.2, 4.7, 5.0, 5.3,
        4.9, 4.4, 4.1, 4.3, 4.8, 5.4, 5.8, 5.5, 6.1, 6.8,
        6.4, 7.1, 7.8, 7.4, 8.2, 8.9, 9.4
    ]
})

df = df.merge(zone_stats, on="zone", how="left")
print(df[["zone", "zone_population_density", "zone_area_km2"]].drop_duplicates().head(10))

    zone  zone_population_density  zone_area_km2
0     15                    23111            2.2
1     22                    31400            1.4
2     67                     8976            5.8
3      1                     8264           10.5
4     60                     9876            5.3
5     24                    14078            4.1
6     71                     8234            6.4
7     49                    11878            4.3
8     26                    12239            4.8
10    32                    16945            3.3


3. ZONE HISTORICAL CRIME RATE

Average daily crimes per zone across all time

In [8]:
zone_historical = (
    df.groupby(["zone", "date"])
    .size()
    .reset_index(name="daily_count")
    .groupby("zone")["daily_count"]
    .mean()
    .reset_index()
    .rename(columns={"daily_count": "zone_historical_crime_rate"})
)

df = df.merge(zone_historical, on="zone", how="left")
print(df[["zone", "zone_historical_crime_rate"]].drop_duplicates().sort_values("zone").head(10))

     zone  zone_historical_crime_rate
3       1                    1.048608
30      2                    1.059187
84      3                    1.053488
82      4                    1.038715
243     5                    1.037017
87      6                    1.103115
228     7                    1.082328
22      8                    1.130827
885     9                    1.018100
413    10                    1.033971


4. ROLLING CRIME RATES

Daily crime count per zone

In [ ]:
daily_zone = (
    df.groupby(["zone", "date"])
    .size()
    .reset_index(name="daily_count")
)

daily_zone["date"] = pd.to_datetime(daily_zone["date"])
daily_zone = daily_zone.sort_values(["zone", "date"])

In [10]:
# Rolling 7-day and 30-day (excluding current day)
daily_zone["rolling_7day_crime_rate"] = (
    daily_zone.groupby("zone")["daily_count"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

daily_zone["rolling_30day_crime_rate"] = (
    daily_zone.groupby("zone")["daily_count"]
    .transform(lambda x: x.shift(1).rolling(30, min_periods=1).mean())
)

In [11]:
# Join back to main df
df["date"] = pd.to_datetime(df["date"])
df = df.merge(
    daily_zone[["zone", "date", "rolling_7day_crime_rate", "rolling_30day_crime_rate"]],
    on=["zone", "date"],
    how="left"
)

print(df[["zone", "date", "rolling_7day_crime_rate", "rolling_30day_crime_rate"]].head(10))

   zone                date  rolling_7day_crime_rate  rolling_30day_crime_rate
0    15 2008-09-15 18:45:00                      NaN                       NaN
1    22 2008-09-15 18:45:00                      NaN                       NaN
2    67 2008-09-15 18:45:00                      NaN                       NaN
3     1 2008-09-15 18:45:00                      NaN                       NaN
4    60 2008-09-15 18:45:00                      NaN                       NaN
5    24 2008-09-15 18:46:00                      NaN                       NaN
6    71 2008-09-15 18:46:00                      NaN                       NaN
7    49 2008-09-15 18:48:46                      NaN                       NaN
8    26 2008-09-15 18:52:14                      NaN                       NaN
9     1 2008-09-15 18:55:00                      1.0                       1.0


In [12]:
print(df["rolling_7day_crime_rate"].isna().sum())
print(df["rolling_30day_crime_rate"].isna().sum())

84
84


In [13]:
df["rolling_7day_crime_rate"] = df["rolling_7day_crime_rate"].fillna(0)
df["rolling_30day_crime_rate"] = df["rolling_30day_crime_rate"].fillna(0)

5. AGGREGATE TO MODEL ROWS

In [16]:
df["date"] = pd.to_datetime(df["date"]).dt.date

agg = (
    df.groupby(["zone", "date", "timeslot"])
    .agg(
        crime_count=("id", "count"),
        hour_bucket=("hour_bucket", "first"),
        day_of_week=("day_of_week", "first"),
        month=("month", "first"),
        year=("year", "first"),
        season=("season", "first"),
        is_weekend=("is_weekend", "first"),
        is_holiday=("is_holiday", "first"),
        zone_population_density=("zone_population_density", "first"),
        zone_area_km2=("zone_area_km2", "first"),
        zone_historical_crime_rate=("zone_historical_crime_rate", "first"),
        rolling_7day_crime_rate=("rolling_7day_crime_rate", "first"),
        rolling_30day_crime_rate=("rolling_30day_crime_rate", "first"),
    )
    .reset_index()
)

print(agg.shape)
print(agg.head())

(1449628, 16)
   zone        date   timeslot  crime_count  hour_bucket  day_of_week  month  \
0     1  2008-09-15    evening            8           16            0      9   
1     1  2008-09-15      night            1           20            0      9   
2     1  2008-09-16  afternoon            4           12            1      9   
3     1  2008-09-16    evening            4           16            1      9   
4     1  2008-09-16    morning            4            8            1      9   

   year season  is_weekend  is_holiday  zone_population_density  \
0  2008   fall           0           0                     8264   
1  2008   fall           0           0                     8264   
2  2008   fall           0           0                     8264   
3  2008   fall           0           0                     8264   
4  2008   fall           0           0                     8264   

   zone_area_km2  zone_historical_crime_rate  rolling_7day_crime_rate  \
0           10.5             

6. RISK SCORE

In [17]:
agg["raw_risk"] = (
    agg["crime_count"] / 
    (agg["zone_population_density"] * agg["zone_area_km2"] * agg["zone_historical_crime_rate"])
)

agg["risk_score"] = (agg["raw_risk"] - agg["raw_risk"].min()) / (agg["raw_risk"].max() - agg["raw_risk"].min())

print(agg["risk_score"].describe())

count    1.449628e+06
mean     1.523893e-02
std      1.532695e-02
min      0.000000e+00
25%      2.678106e-03
50%      1.004595e-02
75%      2.116843e-02
max      1.000000e+00
Name: risk_score, dtype: float64


In [18]:
agg.to_parquet(r"C:\Users\rnch6\OneDrive\Desktop\Programming\UrbanEYE\data\processed\features.parquet", index=False)
print("Saved successfully")
print(agg.shape)

Saved successfully
(1449628, 18)


In [19]:
weather = pd.read_parquet(r"C:\Users\rnch6\OneDrive\Desktop\Programming\UrbanEYE\data\processed\weather_chicago.parquet")
weather["date"] = pd.to_datetime(weather["date"]).dt.date

agg["date"] = pd.to_datetime(agg["date"]).dt.date

agg = agg.merge(weather, on="date", how="left")

print(agg.shape)
print(agg[["date", "temperature", "precipitation", "humidity", "wind_speed"]].head())
print(agg["temperature"].isna().sum())

(1449628, 22)
         date  temperature  precipitation  humidity  wind_speed
0  2008-09-15         15.6            0.0        75        13.2
1  2008-09-15         15.6            0.0        75        13.2
2  2008-09-16         16.2            0.0        66        11.3
3  2008-09-16         16.2            0.0        66        11.3
4  2008-09-16         16.2            0.0        66        11.3
0


In [20]:
agg.to_parquet(r"C:\Users\rnch6\OneDrive\Desktop\Programming\UrbanEYE\data\processed\features.parquet", index=False)
print("Done:", agg.shape)
print(agg.columns.tolist())

Done: (1449628, 22)
['zone', 'date', 'timeslot', 'crime_count', 'hour_bucket', 'day_of_week', 'month', 'year', 'season', 'is_weekend', 'is_holiday', 'zone_population_density', 'zone_area_km2', 'zone_historical_crime_rate', 'rolling_7day_crime_rate', 'rolling_30day_crime_rate', 'raw_risk', 'risk_score', 'temperature', 'precipitation', 'humidity', 'wind_speed']
